# Ingest races.csv file
### 1. Read the file using spark dataframe reader API
### 2. Add Metadata Columns 
-       Source File
-       Ingestion Timestamp
### 3. Write to bronze delta table

### Step 1 - Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
    DateType
)

races_schema = StructType(
    [
        StructField("season", IntegerType()),
        StructField("round", IntegerType()),
        StructField("url", StringType()),
        StructField("raceName", StringType()),
        StructField("date", DateType()),
        StructField("circuitId", StringType())
    ]
)

In [0]:
races_df = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(races_schema)
    .option('mode', 'FAILFAST')
    .load("/Volumes/formula1/landing/files/races.csv")
)

display(races_df)

In [0]:
from pyspark.sql import functions as F

races_final_df = (
    races_df
        .withColumn("Ingestion_Timestamp", F.current_timestamp())
        .withColumn("Source_File", F.col('_metadata.file_path'))
)

In [0]:
display(races_final_df)

In [0]:
(
    races_final_df
    .write
    .format('delta')
    .mode("overwrite")
    .saveAsTable('formula1.bronze.races')
)

In [0]:
%sql
select * from formula1.bronze.races